In [13]:
import os
import pandas as pd
import yaml

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

In [14]:
features_config = yaml.safe_load(open(os.path.join('..', 'src', 'config', 'feature_config.yaml'), "r"))

In [15]:
df_processed = pd.read_csv(os.path.join('..', 'data', 'processed', 'meli_processed.csv'))

print(df_processed.shape)
df_processed.head()

(41176, 23)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,quarter,contacts_tendency
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2Q,0.0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2Q,0.0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2Q,0.0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2Q,0.0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2Q,0.0


In [16]:
month_to_quarter = {
    'jan': '1Q', 'feb': '1Q', 'mar': '1Q',
    'apr': '2Q', 'may': '2Q', 'jun': '2Q',
    'jul': '3Q', 'aug': '3Q', 'sep': '3Q',
    'oct': '4Q', 'nov': '4Q', 'dec': '4Q'
}

df_processed['quarter'] = df_processed['month'].map(month_to_quarter)


In [17]:
not_employed = ['retired', 'student', 'unemployed']

def categorize_employment(job):
    if job in not_employed:
        return 'not_employed'
    elif job == 'unknown':
        return 'unknown'
    else:
        return 'employed'

df_processed['is_employed'] = df_processed['job'].apply(categorize_employment)


In [18]:
def categorizar_faixa_etaria(idade):
    if idade < 18:
        return 'menor de idade'
    elif idade <= 40:
        return 'adulto'
    elif idade <= 60:
        return 'meia idade'
    else:
        return 'idoso'

df_processed['faixa_etaria'] = df_processed['age'].apply(categorizar_faixa_etaria)

df_processed['contatos_anteriores_totais'] = df_processed['campaign'] + df_processed['previous']

df_processed['foi_contatado_antes'] = df_processed['pdays'] != 999

In [19]:
df_processed[['faixa_etaria', 'contatos_anteriores_totais', 'foi_contatado_antes', 'month', 'quarter', 'job', 'is_employed', 'campaign', 'previous']].head()

,faixa_etaria,contatos_anteriores_totais,foi_contatado_antes,month,quarter,job,is_employed,campaign,previous
0,meia idade,1,False,may,2Q,housemaid,employed,1,0
1,meia idade,1,False,may,2Q,services,employed,1,0
2,adulto,1,False,may,2Q,services,employed,1,0
3,adulto,1,False,may,2Q,admin.,employed,1,0
4,meia idade,1,False,may,2Q,services,employed,1,0


In [21]:
df_processed.groupby(['foi_contatado_antes']).size().reset_index(name='count')

,foi_contatado_antes,count
0,False,39661
1,True,1515


In [22]:
df_processed.groupby('job')['y'].value_counts(normalize=True).unstack()


y,no,yes
job,,
admin.,0.870333,0.129667
blue-collar,0.931049,0.068951
entrepreneur,0.914835,0.085165
housemaid,0.900000,0.100000
management,0.887825,0.112175
retired,0.747381,0.252619
self-employed,0.895144,0.104856
services,0.918578,0.081422
student,0.685714,0.314286


In [23]:
df_processed.groupby('is_employed')['y'].value_counts(normalize=True).unstack()


y,no,yes
is_employed,,
employed,0.899326,0.100674
not_employed,0.763515,0.236485
unknown,0.887879,0.112121


In [24]:
pd.crosstab(
    index=[df_processed['loan'], df_processed['housing']],
    columns=df_processed['y'],
    normalize='index'
)


y                      no       yes
loan    housing                    
no      no       0.890958  0.109042
        yes      0.882662  0.117338
unknown unknown  0.891919  0.108081
yes     no       0.892843  0.107157
        yes      0.889190  0.110810